In [20]:
import ast
import pickle

import joblib
import numpy as np
import pandas as pd
from sklearn import linear_model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [21]:
def count_items(value):
    if pd.isna(value):
        return 0

    text = str(value).strip()
    if text in ('', '[]', 'nan', 'None'):
        return 0

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set, dict)):
            return len(parsed)
    except (ValueError, SyntaxError):
        pass

    cleaned = text.strip('[]')
    if not cleaned:
        return 0

    return len([item for item in cleaned.split(',') if item.strip()])

chunk_columns = ['abstract', 'authors', 'n_citation', 'references', 'title', 'venue', 'year', 'id']
chunks = []
for chunk in pd.read_csv('dblp-v10.csv', usecols=chunk_columns, chunksize=50000, engine='python'):
    chunk = chunk.drop_duplicates().copy()
    text_columns = ['abstract', 'authors', 'references', 'title', 'venue']
    chunk[text_columns] = chunk[text_columns].fillna('')
    chunk['year'] = pd.to_numeric(chunk['year'], errors='coerce')
    chunk['n_citation'] = pd.to_numeric(chunk['n_citation'], errors='coerce')
    chunk = chunk.dropna(subset=['year', 'n_citation']).copy()
    chunk['year'] = chunk['year'].astype(int)
    chunk['reference_count'] = chunk['references'].apply(count_items)
    chunk['author_count'] = chunk['authors'].apply(count_items)
    chunk['title_length'] = chunk['title'].str.len()
    chunk['abstract_length'] = chunk['abstract'].str.len()
    chunk['venue_length'] = chunk['venue'].str.len()
    chunks.append(chunk)

clean_df = pd.concat(chunks, ignore_index=True).drop_duplicates().copy()
clean_df['high_impact'] = (clean_df['n_citation'] > clean_df['n_citation'].median()).astype(int)

print('Cleaned rows:', len(clean_df))
print('Duplicate rows:', clean_df.duplicated().sum())
print('Missing values per column:')
print(clean_df.isna().sum())

assert clean_df.isna().sum().sum() == 0
assert clean_df.duplicated().sum() == 0

clean_df.head()

Cleaned rows: 1000000
Duplicate rows: 0
Missing values per column:
abstract           0
authors            0
n_citation         0
references         0
title              0
venue              0
year               0
id                 0
reference_count    0
author_count       0
title_length       0
abstract_length    0
venue_length       0
high_impact        0
dtype: int64


,abstract,authors,n_citation,references,title,venue,year,id,reference_count,author_count,title_length,abstract_length,venue_length,high_impact
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b,7,2,61,511,55,1
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db,3,4,93,1021,14,1
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de,7,2,52,711,35,1
3,One of the fundamental challenges of recognizi...,"['Yaser Sheikh', 'Mumtaz Sheikh', 'Mubarak Shah']",221,"['056116c1-9e7a-4f9b-a918-44eb199e67d6', '05ac...",Exploring the space of a human action,international conference on computer vision,2005,4ab3a98c-3620-47ec-b578-884ecf4a6206,10,3,37,1019,43,1
4,This paper generalizes previous optimal upper ...,"['Efraim Laksman', 'Håkan Lennerstad', 'Magnus...",0,"['01a765b8-0cb3-495c-996f-29c36756b435', '5dbc...",Generalized upper bounds on the minimum distan...,Ima Journal of Mathematical Control and Inform...,2015,4ab3b585-82b4-4207-91dd-b6bce7e27c4e,9,3,67,201,51,0


In [22]:
X = clean_df[['year', 'reference_count', 'author_count', 'title_length', 'abstract_length', 'venue_length']]
y = clean_df['high_impact']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=10,
    stratify=y
)

X_test.head()

,year,reference_count,author_count,title_length,abstract_length,venue_length
920313,2016,2,3,51,1194,48
137836,2007,0,4,56,1023,22
709276,2016,8,3,70,828,48
841433,2016,5,4,67,896,0
454813,2009,14,7,64,415,39


In [23]:
X = clean_df[['year', 'reference_count', 'author_count', 'title_length', 'abstract_length', 'venue_length']]
y = clean_df['high_impact']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=10,
    stratify=y
)

X_test.head()

,year,reference_count,author_count,title_length,abstract_length,venue_length
920313,2016,2,3,51,1194,48
137836,2007,0,4,56,1023,22
709276,2016,8,3,70,828,48
841433,2016,5,4,67,896,0
454813,2009,14,7,64,415,39


In [24]:
model = LogisticRegression(max_iter=2000, C=0.1)
model.fit(X_train, y_train)
model.coef_

array([[-0.15899244,  0.04684668,  0.08622526, -0.00316388,  0.0006442 ,
         0.00574855]])

In [25]:
model.intercept_

array([318.34477154])

Save Model To a File Using Python Pickle

In [26]:
with open('model_pickle', 'wb') as file:
    pickle.dump(model, file)

Load Saved Model

In [27]:
with open('model_pickle', 'rb') as file:
    mp = pickle.load(file)

In [28]:
sample_input = pd.DataFrame({
    'year': [2015],
    'reference_count': [15],
    'author_count': [4],
    'title_length': [55],
    'abstract_length': [850],
    'venue_length': [42],
})

sample_prediction = model.predict(sample_input)
sample_probability = model.predict_proba(sample_input)[:, 1]

sample_prediction

array([0])

In [29]:
sample_probability

array([0.41037854])

In [30]:
best_threshold = 0.195
sample_label = int(sample_probability[0] >= best_threshold)
sample_label

1

In [31]:
with open('model_pickle','rb') as file:
    mp = pickle.load(file)

Save Model To a File Using Python Pickle

In [32]:
with open('model_pickle','wb') as file:
    pickle.dump(model,file)

Load Saved Model

In [33]:
mj = joblib.load('model_joblib')

In [34]:
mp.intercept_

array([318.34477154])

In [35]:
int(mp.predict_proba(sample_input)[:, 1][0] >= best_threshold)

1

In [36]:
mj.predict(sample_input)

array([0])

In [37]:
best_threshold = 0.195
test_probabilities = model.predict_proba(X_test)[:, 1]
threshold_predictions = (test_probabilities >= best_threshold).astype(int)

print('Best threshold:', best_threshold)
print('Accuracy at best threshold:', accuracy_score(y_test, threshold_predictions))
print('Confusion matrix at best threshold:')
print(confusion_matrix(y_test, threshold_predictions))
print('Classification report at best threshold:')
print(classification_report(y_test, threshold_predictions))

Best threshold: 0.195
Accuracy at best threshold: 0.5738533333333333
Confusion matrix at best threshold:
[[ 27702 124147]
 [  3697 144454]]
Classification report at best threshold:
              precision    recall  f1-score   support

           0       0.88      0.18      0.30    151849
           1       0.54      0.98      0.69    148151

    accuracy                           0.57    300000
   macro avg       0.71      0.58      0.50    300000
weighted avg       0.71      0.57      0.50    300000

